In [1]:
import csv
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

/var/folders/59/mg36zpzn3fqbx4_xccrmb89m0000gn/T/ipykernel_46973/4292752259.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import pandas as pd
import numpy as np

In [4]:
books = pd.read_csv("data/books_cleaned.csv")


In [5]:
books.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."


In [6]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: str

In [7]:
books["tagged_description"].to_csv(
    "data/tagged_description.txt",
    index=False,
    header=False,
    quoting=csv.QUOTE_NONE,
    escapechar="\\",
)

In [8]:
raw_documents = TextLoader("data/tagged_description.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=10000, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)

In [9]:
documents[0]

Document(metadata={'source': 'data/tagged_description.txt'}, page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade\\, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher\\, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead\\, Iowa\\, towards the end of the Reverend Ames’s life\\, and he is absorbed in recording his family’s story\\, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence\\, John tells of the rift between his grandfather and his father: the elder\\, an angry visionary who fought for the abolitionist cause\\, and his son\\, an ardent pacifist. He is troubled\\, too\\, by his prodigal namesake\\, Jack (John Ames) Boughton\\, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous\\, rambling voice that finds beauty\\, humour and truth 

In [10]:
db_books = Chroma.from_documents(
    documents,
    embedding = OpenAIEmbeddings()
)

In [11]:
query = "A book to teach childrean about nature"

docs = db_books.similarity_search(query,k = 10)
docs

[Document(id='61814b09-e7c5-4bcc-a5de-b271c7e6f1b5', metadata={'source': 'data/tagged_description.txt'}, page_content='9780060852559 Bestselling author Barbara Kingsolver returns with her first nonfiction narrative that will open your eyes in a hundred new ways to an old truth: You are what you eat. \\"As the U.S. population made an unprecedented mad dash for the Sun Belt\\, one carload of us paddled against the tide\\, heading for the Promised Land where water falls from the sky and green stuff grows all around. We were about to begin the adventure of realigning our lives with our food chain. \\"Naturally\\, our first stop was to buy junk food and fossil fuel. . . .\\" Hang on for the ride: With characteristic poetry and pluck\\, Barbara Kingsolver and her family sweep readers along on their journey away from the industrial-food pipeline to a rural life in which they vow to buy only food raised in their own neighborhood\\, grow it themselves\\, or learn to live without it. Their good-

In [12]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
260,9780060852559,0060852550,"Animal, Vegetable, Miracle",Barbara Kingsolver;Camille Kingsolver;Steven L...,Biography & Autobiography,http://books.google.com/books/content?id=qLkEY...,Bestselling author Barbara Kingsolver returns ...,2007.0,4.04,370.0,86130.0,"Animal, Vegetable, Miracle: A Year of Food Life",9780060852559 Bestselling author Barbara Kings...


In [13]:
def retrieve_semantic_recommendations(query: str, top_k: int = 10):
    recs = db_books.similarity_search(query, k=50)

    books_list = []
    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)].head(top_k)

In [15]:
retrieve_semantic_recommendations("A book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
86,9780060000141,0060000147,Poppy's Return,Avi,Juvenile Fiction,http://books.google.com/books/content?id=XbcMJ...,"There's trouble at Gray House, the girlhood ho...",2006.0,3.99,256.0,1086.0,Poppy's Return,"9780060000141 There's trouble at Gray House, t..."
101,9780060175641,0060175648,Identity,Milan Kundera,Fiction,http://books.google.com/books/content?id=D30Ex...,Milan Kundera's Identity translated from the F...,1998.0,3.68,176.0,260.0,Identity: A Novel,9780060175641 Milan Kundera's Identity transla...
114,9780060509057,0060509058,Travels,Michael Crichton,Biography & Autobiography,http://books.google.com/books/content?id=QYilZ...,Often I feel I go to some distant region of th...,2002.0,3.95,400.0,6812.0,Travels,9780060509057 Often I feel I go to some distan...
203,9780060736255,0060736259,Weetzie Bat,Francesca Lia Block,Juvenile Fiction,http://books.google.com/books/content?id=vxCXx...,Fifteen years ago Francesca Lia Block made a d...,2004.0,3.74,128.0,12771.0,Weetzie Bat,9780060736255 Fifteen years ago Francesca Lia ...
218,9780060763725,0060763728,Psyche in a Dress,Francesca Lia Block,Juvenile Fiction,http://books.google.com/books/content?id=NYE3G...,But this is what I could not give up: I could ...,2006.0,3.84,116.0,2414.0,Psyche in a Dress,9780060763725 But this is what I could not giv...
260,9780060852559,0060852550,"Animal, Vegetable, Miracle",Barbara Kingsolver;Camille Kingsolver;Steven L...,Biography & Autobiography,http://books.google.com/books/content?id=qLkEY...,Bestselling author Barbara Kingsolver returns ...,2007.0,4.04,370.0,86130.0,"Animal, Vegetable, Miracle: A Year of Food Life",9780060852559 Bestselling author Barbara Kings...
306,9780060932664,006093266X,Collected Novellas,Gabriel Garcia Marquez,Fiction,http://books.google.com/books/content?id=JRcVu...,"Renowned as a master of magical realism, Gabri...",1999.0,4.01,288.0,822.0,Collected Novellas,9780060932664 Renowned as a master of magical ...
363,9780061120077,0061120073,A Tree Grows in Brooklyn,Betty Smith,Fiction,http://books.google.com/books/content?id=Y-FZ9...,The beloved American classic about a young gir...,2006.0,4.26,496.0,326733.0,A Tree Grows in Brooklyn,9780061120077 The beloved American classic abo...
388,9780061196676,0061196673,Smithsonian Intimate Guide to Human Origins,Carl Zimmer,Social Science,http://books.google.com/books/content?id=xufuS...,From the savannas of Africa to modern-day labs...,2007.0,4.00,176.0,167.0,Smithsonian Intimate Guide to Human Origins,9780061196676 From the savannas of Africa to m...
406,9780064403870,0064403874,"R-T, Margaret, and the Rats of NIMH",Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=WTHHH...,"When Margaret and her younger brother, Artie, ...",1991.0,3.52,272.0,631.0,"R-T, Margaret, and the Rats of NIMH",9780064403870 When Margaret and her younger br...
